# Logistics AI Intelligence — Final Project Notebook

**CodeSearchNet-inspired transportation and supply-chain intelligence pipeline**

This notebook mirrors the progression of the IT559 CodeSearchNet final project: problem definition → acquisition → cleaning → EDA → feature engineering → model training → evaluation → retrieval/inference → limitations and future work.

## 1. Problem Definition and Research Questions

**Task 1 — Predictive intelligence:** predict whether a shipment/route is delayed and, where possible, estimate delivery-time deviation.

**Task 2 — Semantic retrieval:** retrieve relevant logistics records from natural-language operational questions.

**Task 3 — Smart routing:** rank candidate routes/lanes using delay, risk, congestion, weather, cost, distance, and reliability signals that actually exist in the available data.

**RQ1:** Which operational variables best explain delay/risk?  
**RQ2:** How well do classical baselines generalize on held-out records?  
**RQ3:** Can TF-IDF retrieval provide useful evidence for analyst questions?  
**RQ4:** Can a transparent multi-factor score produce explainable candidate-route recommendations?

In [ ]:
from pathlib import Path
import os, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
os.environ.setdefault('PYTHONPATH',str(ROOT))
print(ROOT)

## 2. Data Acquisition

The primary workflow uses the three Kaggle datasets documented in `docs/DATASETS.md`. Run the downloader once, then all downstream cells operate locally and reproducibly.

In [ ]:
# Uncomment when Kaggle access is configured:
# %run ../scripts/download_kaggle.py
from backend.app.core.config import settings
print('Raw data directory:',settings.raw_dir)

## 3. Unified Schema and Preprocessing

Each source is mapped into a canonical logistics schema. Source-specific values are preserved where possible; missing concepts remain null rather than being fabricated.

In [ ]:
from backend.app.data.adapters import discover_and_load, DynamicSupplyChainAdapter
from backend.app.data.demo import generate_demo_dataset
if settings.data_mode=='demo':
    p=generate_demo_dataset(settings.demo_dir/'dynamic_supply_chain_logistics_dataset.csv')
    df=DynamicSupplyChainAdapter(source_name='SYNTHETIC_DEMO_FIXTURE').load(p)
else:
    df=discover_and_load(settings.raw_dir)
print(df.shape)
display(df.head())

## 4. Exploratory Data Analysis

Replicates the CodeSearchNet project’s descriptive-statistics and figure-first workflow, but uses operational logistics variables.

In [ ]:
display(df[['distance_km','delay_hours','traffic_level','weather_severity','shipping_cost_usd','route_risk']].describe())
df.source_dataset.value_counts().plot(kind='bar',title='Records by Source Dataset'); plt.show()
if df.delay_hours.notna().any():
    df.delay_hours.dropna().clip(upper=df.delay_hours.quantile(.99)).hist(bins=35); plt.title('Delay Hours'); plt.show()

## 5. Feature Engineering and Model Training

Numeric and categorical operational variables are imputed and encoded inside scikit-learn pipelines. Logistic Regression and Random Forest classifiers are compared on a held-out split, mirroring the classical-model comparison discipline of the CodeSearchNet project.

In [ ]:
from backend.app.ml.training import train_all
metrics=train_all(df)
metrics

## 6. Semantic Retrieval — Logistics Equivalent of `predict_topk`

A TF-IDF word/bigram index is built over human-readable logistics record narratives. Queries and records occupy the same feature space and are ranked with cosine similarity.

In [ ]:
from backend.app.ml.training import retrieve
retrieve('high congestion route with elevated delay risk',k=5)

## 7. Dataset-Grounded Q&A

In [ ]:
from backend.app.services.insights import answer
answer('What do the relevant records say about delays and route risk?',k=8)

## 8. Smart Route Ranking

The ranker is deliberately explainable. It uses only supplied fields and reports the factors used; it does not claim live traffic/weather data unless those values are supplied by a live integration.

In [ ]:
from backend.app.services.insights import route_recommend
route_recommend([
 {'route_id':'A','distance_km':450,'traffic_level':7,'weather_severity':.4,'route_risk':6,'delay_probability':.65,'shipping_cost_usd':980},
 {'route_id':'B','distance_km':500,'traffic_level':3,'weather_severity':.15,'route_risk':3,'delay_probability':.25,'shipping_cost_usd':1080},
 {'route_id':'C','distance_km':430,'traffic_level':8,'weather_severity':.7,'route_risk':8,'delay_probability':.8,'shipping_cost_usd':900}
])

## 9. Qualitative Error Analysis

Inspect false positives/false negatives from the classifier and retrieval near-misses. For logistics, ambiguous failures typically reflect missing operational context, sparse route observations, data-source differences, and the fact that historical traffic/weather is not equivalent to current conditions.

In [ ]:
from pathlib import Path
print('Generated figures:')
for p in sorted(settings.figures_dir.glob('*.png')): print(' -',p.name)
print(json.dumps(metrics,indent=2,default=str)[:5000])

## 10. Limitations and Future Work

1. These datasets are historical; not all are real-time and some are synthetic by design.
2. The normalized trucking database explicitly omits traffic/construction and weather impacts.
3. Candidate ranking is not a road-network shortest-path solver.
4. TF-IDF is lexical and should eventually be compared with sentence-transformer/embedding retrieval.
5. Production deployment should add live map/traffic/weather APIs, calibration, drift monitoring, authenticated users, persistent storage, and human dispatch approval.

## Conclusion

The resulting system extends the CodeSearchNet methodology from semantic code understanding into a complete logistics intelligence application: reproducible data ingestion, EDA/figures, predictive ML, semantic retrieval, evidence-grounded chat, explainable smart routing, FastAPI services, and a Vue 3 frontend.